# 02 · 单 Batch 真正反向更新(★★★★★)

从「会算损失」到「模型真的学了一步」。**验证 6 件事**:学生有梯度、教师梯度 None、学生权重变、教师权重不变、四项损失都反传、loss 有限。

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


## 1) 教师(冻结)+ 学生(4.35M)

In [ ]:
from onescience.models.pangu import Pangu
from maxvit3d_student import MaxVit3DStudent
import maxvit3d_student as M; M.set_sdpa(True)
dev = 0
teacher = Pangu(img_size=(721,1440)).to(dev).eval()
ck = torch.load(CKPT, map_location=f"cuda:{dev}", weights_only=False)
teacher.load_state_dict(ck["model_state_dict"])
for p in teacher.parameters(): p.requires_grad_(False)   # ★教师冻结
student = MaxVit3DStudent(patch_size=(2,16,16), embed_dim=96, depths=(2,4,2),
                          num_heads=(6,12,6), mlp_ratio=2.0).to(dev)
opt = torch.optim.AdamW(student.parameters(), lr=6e-4)
print("学生参数:", sum(p.numel() for p in student.parameters())/1e6, "M")

## 2) 一个输入 + 双监督加权 L1(教师输出 + 真值)

In [ ]:
x = torch.randn(1, 72, 721, 1440, device=dev)     # 实战用 01 的 x72;这里随机演示
sw = torch.tensor([1.5,0.77,0.66,3.0], device=dev).view(1,4,1,1)   # 面通道权(config weights[:4])
pw = torch.ones(1,65,1,1, device=dev)                               # 高空权
def wl1(a,b,w,lw): return lw * (F.l1_loss(a,b,reduction="none")*w).mean()

w_before = student.enc[0].mlp.w3.weight.detach().clone()   # 记一个学生权重
t_before = next(teacher.parameters()).detach().clone()     # 记一个教师权重
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    ts_, tu_ = teacher(x); tu_ = tu_.reshape(1,65,721,1440)          # 教师输出(常量)
with torch.autocast("cuda", dtype=torch.bfloat16):
    ss, su = student(x); su = su.reshape(1,65,721,1440)              # 学生输出
pred = torch.cat([ss.float(), su.float()], 1)
loss = wl1(pred[:,:4], ts_.float(), sw, 0.25) + wl1(pred[:,4:], tu_.float(), pw, 1.0)  # 对教师
print("loss =", round(loss.item(),4))

## 3) ★真正反向一步 + 6 项验证(全 True 才算学了一步)

In [ ]:
opt.zero_grad()
loss.backward()
gnorm = torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)   # 梯度裁剪
opt.step()
w_after = student.enc[0].mlp.w3.weight.detach()
t_after = next(teacher.parameters()).detach()
assert torch.isfinite(loss),                    "loss 必须有限"
assert student.enc[0].mlp.w3.weight.grad is not None, "学生要有梯度"
assert all(p.grad is None for p in teacher.parameters()), "教师梯度必须 None"
assert not torch.allclose(w_before, w_after),   "学生权重必须变了"
assert torch.allclose(t_before, t_after),       "教师权重必须没变"
print("✅ 全部通过 | loss=%.4f 梯度范数=%.3f" % (loss.item(), gnorm))
print("  学生权重变化量:", (w_after-w_before).abs().mean().item())

### ✅ 实测结论:loss 有限、学生有梯度且权重更新、教师梯度 None 且权重不变 —— 这就是「学了一步」。